In [1]:
import pandas as pd

with open("../data/raw/OpenSSH_2k.log") as f:
    lines = f.read().splitlines()

df = pd.DataFrame({"line": lines})
print(len(df))
df.head(10)

2000


,line
0,Dec 10 06:55:46 LabSZ sshd[24200]: reverse map...
1,Dec 10 06:55:46 LabSZ sshd[24200]: Invalid use...
2,Dec 10 06:55:46 LabSZ sshd[24200]: input_usera...
3,Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(ss...
4,Dec 10 06:55:46 LabSZ sshd[24200]: pam_unix(ss...
5,Dec 10 06:55:48 LabSZ sshd[24200]: Failed pass...
6,Dec 10 06:55:48 LabSZ sshd[24200]: Connection ...
7,Dec 10 07:02:47 LabSZ sshd[24203]: Connection ...
8,Dec 10 07:07:38 LabSZ sshd[24206]: Invalid use...
9,Dec 10 07:07:38 LabSZ sshd[24206]: input_usera...


In [2]:
patterns = ["Failed password", "Invalid user", "Accepted password", "Connection closed"]
for p in patterns:
    print(p, df["line"].str.contains(p).sum())

Failed password 520
Invalid user 113
Accepted password 1
Connection closed 34


In [3]:
failed = df[df["line"].str.contains("Failed password")]
ips = failed["line"].str.extract(r"from (\d+\.\d+\.\d+\.\d+)")
ips[0].value_counts().head(10)

0
183.62.140.253     286
187.141.143.180     80
103.99.0.122        46
112.95.230.3        26
5.188.10.180        18
185.190.58.151      17
123.235.32.19        7
119.4.203.64         6
52.80.34.196         5
60.2.12.12           5
Name: count, dtype: int64

In [1]:
import sys; sys.path.append("../src")
from parsers import parse_file

df = parse_file("../data/raw/OpenSSH_2k.log")
print(len(df))
df["event_type"].value_counts()

666


event_type
failed_password        385
failed_invalid_user    134
invalid_user           112
connection_closed       34
accepted_password        1
Name: count, dtype: int64

In [2]:
import sys; sys.path.append("../src")
from parsers import parse_file
from features import build_features
from detectors import detect_rules

df = parse_file("../data/raw/OpenSSH_2k.log")
print(len(df)); print(df["event_type"].value_counts())

feats = build_features(df)
print(feats.describe())          # look at the real numbers before choosing thresholds

alerts = detect_rules(feats)
print(len(alerts))
alerts.head(10)

666
event_type
failed_password        385
failed_invalid_user    134
invalid_user           112
connection_closed       34
accepted_password        1
Name: count, dtype: int64
                           window      failed    invalid   accepted  \
count                          49   49.000000  49.000000  49.000000   
mean   2015-12-10 09:00:36.734694   10.591837   2.285714   0.020408   
min           2015-12-10 06:55:00    0.000000   0.000000   0.000000   
25%           2015-12-10 07:55:00    1.000000   0.000000   0.000000   
50%           2015-12-10 09:10:00    1.000000   1.000000   0.000000   
75%           2015-12-10 10:10:00    6.000000   1.000000   0.000000   
max           2015-12-10 11:00:00  141.000000  28.000000   1.000000   
std                           NaN   27.834270   5.291503   0.142857   

       unique_users      events  
count     49.000000   49.000000  
mean       2.448980   13.591837  
min        0.000000    1.000000  
25%        1.000000    2.000000  
50%        1.0

,alert_id,source_ip,window_start,failed_logins,unique_usernames,successful_logins,detector
0,A-001,103.99.0.122,2015-12-10 09:10:00,30,19,0,brute_force
1,A-002,103.99.0.122,2015-12-10 09:10:00,30,19,0,user_enumeration
2,A-003,103.99.0.122,2015-12-10 11:00:00,16,12,0,brute_force
3,A-004,103.99.0.122,2015-12-10 11:00:00,16,12,0,user_enumeration
4,A-005,112.95.230.3,2015-12-10 07:25:00,26,3,0,brute_force
5,A-006,183.62.140.253,2015-12-10 10:50:00,16,3,0,brute_force
6,A-007,183.62.140.253,2015-12-10 10:55:00,141,8,0,brute_force
7,A-008,183.62.140.253,2015-12-10 10:55:00,141,8,0,user_enumeration
8,A-009,183.62.140.253,2015-12-10 11:00:00,129,1,0,brute_force
9,A-010,185.190.58.151,2015-12-10 09:10:00,11,2,0,brute_force


In [3]:
df.to_csv("../data/processed/ssh_parsed.csv", index=False)
alerts.to_json("../data/processed/alerts.json", orient="records", indent=2)